**Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Is the stock market overvalued? Part 1: getting and cleaning data

So far, we've found data, cleaned it in Excel, uploaded it, and charted it, all by hand. Now we'll do the same steps **in code**, so they can be repeated and checked.

**By the end you will have:**
1. Fetched files from the internet with Python
2. Calculated a new column (the price-to-earnings ratio)
3. Removed missing data
4. Fixed the dates to `yyyy-mm-dd`, and caught a sneaky bug
5. Saved a clean CSV, ready to chart

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.
>
> Run a cell by clicking it and pressing **Shift + Enter**.

## What is a notebook?

A notebook mixes two kinds of cell:
- **Text cells** (like this one): explanations, headings and links
- **Code cells**: Python you can run, with the result shown underneath

We run code in small steps and check each result before moving on. We use **Google Colab**, which runs Python in your browser with nothing to install.

<br>

## 0. Load our packages

In [10]:
import requests        # fetch files from the internet
import pandas as pd     # work with tables of data

<br>

---

<br>

## 1. Fetching from the internet with `requests`

`requests.get(url)` downloads whatever is at a web address. Let's fetch a Vega-Lite chart from Richard's chart library.

In [11]:
requests.get()

TypeError: get() missing 1 required positional argument: 'url'

In [12]:
chart_url = "https://raw.githubusercontent.com/RDeconomist/RDeconomist.github.io/refs/heads/main/charts/library/chartLine3.json"

chart_response = requests.get(chart_url)
chart_response

<Response [200]>

`<Response [200]>` means it worked (200 is the web's code for "OK").

The file is JSON, so `.json()` turns it into a Python **dictionary**:

In [13]:
chart_spec = chart_response.json()
chart_spec

{'$schema': 'https://vega.github.io/schema/vega-lite/v5.json',
 'title': {'text': 'Declining populations',
  'subtitle': 'Cumulative % change in population. Source: UN.',
  'subtitleFontStyle': 'italic',
  'subtitleFontSize': 10,
  'anchor': 'start',
  'color': 'black'},
 'data': {'url': 'https://raw.githubusercontent.com/RDeconomist/RDeconomist.github.io/main/data/demographicsUNPopChange.csv'},
 'repeat': {'layer': ['S Korea',
   'Japan',
   'UK',
   'Italy',
   'Spain',
   'France',
   'US']},
 'spec': {'height': 300,
  'width': 240,
  'mark': {'type': 'line', 'strokeWidth': 2},
  'encoding': {'x': {'field': 'Year', 'type': 'temporal'},
   'y': {'field': {'repeat': 'layer'}, 'type': 'quantitative', 'title': None},
   'color': {'datum': {'repeat': 'layer'},
    'scale': {'range': ['black',
      'yellow',
      'orange',
      'lightgrey',
      'lightblue',
      'pink',
      'lightgreen']}}}}}

This is the same Vega-Lite JSON you've been editing. Because it's now a dictionary, we can read parts of it with keys:

In [14]:
chart_spec["title"]

{'text': 'Declining populations',
 'subtitle': 'Cumulative % change in population. Source: UN.',
 'subtitleFontStyle': 'italic',
 'subtitleFontSize': 10,
 'anchor': 'start',
 'color': 'black'}

> **Why this matters:** many data sources (APIs) send back JSON like this. We'll use exactly this technique to fetch live data in Week 4.

<br><br>

---

<br>

## 2. Loading a CSV into a table

Now, some data. We'll use Robert Shiller's long-run data on US stock prices and company earnings, starting in 1871 ([shillerdata.com](https://shillerdata.com/)). A copy is stored as a CSV on GitHub.

For CSV files, `pandas` can read straight from the raw GitHub link:

In [19]:
csv_url = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week03/data/data_shiller.csv"

our_data = pd.read_csv(csv_url, dtype={'Date': str})   # read the Date column as text (more on this in step 5)

In [20]:
our_data

,Date,P,D,E,CPI,Rate GS10,Price_real,Earnings_real,CAPE
0,1871.01,4.440000,0.2600,0.4000,12.464061,5.320000,119.611709,10.775830,NaN
1,1871.02,4.500000,0.2600,0.4000,12.844641,5.323333,117.636158,10.456547,NaN
2,1871.03,4.610000,0.2600,0.4000,13.034972,5.326667,118.752052,10.303866,NaN
3,1871.04,4.740000,0.2600,0.4000,12.559226,5.330000,126.726006,10.694178,NaN
4,1871.05,4.860000,0.2600,0.4000,12.273812,5.333333,132.955747,10.942860,NaN
...,...,...,...,...,...,...,...,...,...
1865,2026.06,7450.032857,81.7032,295.3881,333.952000,4.470000,7490.735069,297.001912,40.150170
1866,2026.07,7481.339091,NaN,NaN,333.918000,4.600000,7522.978262,NaN,40.008636
1867,2026.08,7711.323333,NaN,NaN,334.980000,4.680000,7729.658962,NaN,40.987394
1868,2026.09,7669.414286,NaN,NaN,335.511000,4.990000,7675.483325,NaN,40.580874


`df` (short for **D**ata**F**rame) is now a table, like a sheet in Excel. `.head()` shows the first 5 rows:

In [22]:
df.head(10)

,Date,P,D,E,CPI,Rate GS10,Price_real,Earnings_real,CAPE
0,1871.01,4.44,0.26,0.4,12.464061,5.320000,119.611709,10.775830,NaN
1,1871.02,4.50,0.26,0.4,12.844641,5.323333,117.636158,10.456547,NaN
2,1871.03,4.61,0.26,0.4,13.034972,5.326667,118.752052,10.303866,NaN
3,1871.04,4.74,0.26,0.4,12.559226,5.330000,126.726006,10.694178,NaN
4,1871.05,4.86,0.26,0.4,12.273812,5.333333,132.955747,10.942860,NaN
5,1871.06,4.82,0.26,0.4,12.083481,5.336667,133.938451,11.115224,NaN
6,1871.07,4.73,0.26,0.4,12.083481,5.340000,131.437526,11.115224,NaN
7,1871.08,4.79,0.26,0.4,11.893231,5.343333,135.234015,11.293028,NaN
8,1871.09,4.84,0.26,0.4,12.178646,5.346667,133.443260,11.028369,NaN
9,1871.10,4.59,0.26,0.4,12.368896,5.350000,124.604019,10.858738,NaN


A few useful checks on any new dataset:

In [23]:
df.shape      # (rows, columns)

(1870, 9)

In [26]:
df.columns    # the column names

Index(['Date', 'P', 'D', 'E', 'CPI', 'Rate GS10', 'Price_real',
       'Earnings_real', 'CAPE'],
      dtype='object')

The columns we need:

| Column | Meaning |
|---|---|
| `Date` | Year and month, written as `1871.01` (October is `1871.10`) |
| `P` | Stock price index (S&P composite) |
| `E` | Company earnings |

Pick out one column with `df['column name']`:

In [29]:
df["CPI"]

,CPI
0,12.464061
1,12.844641
2,13.034972
3,12.559226
4,12.273812
...,...
1865,333.952000
1866,333.918000
1867,334.980000
1868,335.511000


In [25]:
df['P']

,P
0,4.440000
1,4.500000
2,4.610000
3,4.740000
4,4.860000
...,...
1865,7450.032857
1866,7481.339091
1867,7711.323333
1868,7669.414286


<details>
<summary><b>Optional: loading a CSV with <code>requests</code> instead</b></summary>

`pd.read_csv(url)` works for simple links like raw GitHub files. For some websites you need to fetch the file first with `requests`, then read it:

```python
import io
response = requests.get(csv_url)
df = pd.read_csv(io.StringIO(response.text))
```
</details>

<br><br>

---

<br>

## 3. Calculating a new column

The **price-to-earnings (P/E) ratio** is the stock price divided by company earnings. A high P/E means investors are paying a lot for each £1 (or $1) of profit, which can be a sign the market is overvalued.

Python does maths on a whole column at once:

In [35]:
df['P'] / df['E']

,0
0,11.100000
1,11.250000
2,11.525000
3,11.850000
4,12.150000
...,...
1865,25.221168
1866,NaN
1867,NaN
1868,NaN


To keep the result, store it as a new column. Choose the name in the square brackets:

In [34]:
df['PE_ratio'] = df['P'] / df['E']

df.tail()   # .tail() shows the last 5 rows

,Date,P,D,E,CPI,Rate GS10,Price_real,Earnings_real,CAPE,PE_ratio
1865,2026.06,7450.032857,81.7032,295.3881,333.9520,4.47,7490.735069,297.001912,40.150170,25.221168
1866,2026.07,7481.339091,NaN,NaN,333.9180,4.60,7522.978262,NaN,40.008636,NaN
1867,2026.08,7711.323333,NaN,NaN,334.9800,4.68,7729.658962,NaN,40.987394,NaN
1868,2026.09,7669.414286,NaN,NaN,335.5110,4.99,7675.483325,NaN,40.580874,NaN
1869,2026.10,7722.720000,NaN,NaN,335.7765,5.24,7722.720000,NaN,40.710609,NaN


<br><br>

---

<br>

## 4. Missing data

The last rows show `NaN` ("not a number"). That means **missing**: the most recent earnings haven't been published yet, so there's no P/E to calculate.

How many are missing?

In [36]:
df['PE_ratio'].isna().sum()

np.int64(4)

In [38]:
print(df['PE_ratio'].isna().sum())

4


`.dropna()` removes rows with missing values. `subset=` says which column to check.

We store the result as a new table, `df_clean`, so the original `df` is still there if we need it. (`.copy()` tells pandas this is a separate table.)

In [37]:
df_clean = df.dropna(subset=['PE_ratio']).copy()

df_clean.tail()

,Date,P,D,E,CPI,Rate GS10,Price_real,Earnings_real,CAPE,PE_ratio
1861,2026.02,6893.806316,80.083533,254.693333,326.785,4.13,7083.489623,261.701229,39.013997,27.067086
1862,2026.03,6654.419091,80.365300,261.723000,330.213,4.25,6766.534182,266.132566,37.031611,25.425427
1863,2026.04,6957.007619,80.811267,272.944700,333.020,4.32,7014.592724,275.203940,38.138278,25.488707
1864,2026.05,7412.551500,81.257233,284.166400,335.123,4.48,7427.006200,284.720533,40.101359,26.085250
1865,2026.06,7450.032857,81.703200,295.388100,333.952,4.47,7490.735069,297.001912,40.150170,25.221168


In [39]:
# Check: no missing values left
df_clean['PE_ratio'].isna().sum()

np.int64(0)

<br><br>

---

<br>

## 5. Fixing the dates

Vega-Lite wants dates as `yyyy-mm-dd`. Ours look like `1871.01` (year.month).

`pd.to_datetime()` converts text into dates. We tell it the current pattern with `format=`:
- `%Y` is a 4-digit year
- `%m` is a 2-digit month

(These are the same date codes Vega-Lite uses.)

In [40]:
df_clean['Date_iso'] = pd.to_datetime(df_clean['Date'], format='%Y.%m')

df_clean[['Date', 'Date_iso']].head(12)

,Date,Date_iso
0,1871.01,1871-01-01
1,1871.02,1871-02-01
2,1871.03,1871-03-01
3,1871.04,1871-04-01
4,1871.05,1871-05-01
5,1871.06,1871-06-01
6,1871.07,1871-07-01
7,1871.08,1871-08-01
8,1871.09,1871-09-01
9,1871.10,1871-10-01


**Always check.** Monthly data should have exactly one row per month, so no date should appear twice. `.duplicated().sum()` counts repeats:

In [41]:
df_clean['Date_iso'].duplicated().sum()

np.int64(0)

<br>

### Why did we load `Date` as text?

When we loaded the data, we added `dtype={'Date': str}`. This tells pandas: *read the Date column as text, exactly as it is written*.

Without it, pandas reads dates as **numbers**. As a number, `1871.10` (October) is the same as `1871.1`, so the zero is dropped:

In [ ]:
pd.read_csv(csv_url)['Date'][9]    # row 9 is October 1871

Read as a date, `1871.1` means month **1**: every October would become January!

This kind of bug is easy to miss: there's no error, and the chart still draws. Two habits catch it:
1. Be careful when numbers and text look alike (remember `"120"` vs `120` from the quick start).
2. Check your data after each step.

<br><br>

---

<br>

## 6. Save a clean CSV

We only need two columns for our chart. Keeping just these makes the file smaller and easier to use.

In [42]:
df_out = df_clean[['Date_iso', 'PE_ratio']]

df_out.to_csv('data_shiller_pe.csv', index=False)   # index=False: don't save pandas' row numbers

df_out.head()

,Date_iso,PE_ratio
0,1871-01-01,11.100
1,1871-02-01,11.250
2,1871-03-01,11.525
3,1871-04-01,11.850
4,1871-05-01,12.150


The file is saved in Colab's **Files** panel (the folder icon on the left).

> **Colab files are temporary.** They are deleted when your session ends, so download the file now:

In [43]:
from google.colab import files     # only exists in Colab
files.download('data_shiller_pe.csv')
# (If you're not running in Colab, you don't need this step)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<br>

### Next steps

1. Upload `data_shiller_pe.csv` to your GitHub repository (just like in Week 2).
2. Open it on GitHub, click **Raw**, and copy the link.

In the seminar, we'll use that link to chart the data with Python (**DS3_2_Altair_Shiller**).

<br><br><br>

---

<br>

## <font color='green'>Stretch exercises</font>

**1.** What was the highest P/E ratio, and when? *Hint:* `.max()`, and `df_clean.sort_values('PE_ratio')`.

In [ ]:
# Your code here


**2.** Calculate the **average** P/E ratio over the whole period with `.mean()`. Is today's value above or below it?

In [ ]:
# Your code here


**3.** The data also has `D` (dividends). Calculate a **dividend yield** column: `D / P * 100`.

In [ ]:
# Your code here
